# Model

In [ ]:
from chapter import *

Combining convolution and pooling layers into a deep model:

In [ ]:
%%save
import torchsummary

mnist_model = lambda: nn.Sequential(
    nn.Conv2d(1, 32, 3, 1, 1),
    nn.SELU(),
    nn.MaxPool2d(2, 2),
    
    nn.Conv2d(32, 32, 5, 1, 0),
    nn.SELU(),
    nn.MaxPool2d(2, 2),
    
    nn.Flatten(),
    nn.Linear(800, 256), nn.SELU(), nn.Dropout(0.5),
    nn.Linear(256, 10)
)

In [ ]:
torchsummary.summary(mnist_model(), (1, 28, 28))

**Remark.** We use **SELU activation** {cite}`selu` for fun. Note that we also used **Dropout** {cite}`dropout` as regularization for the dense layers. Observe that number of parameters due to convolutions is small relative to the final network size (i.e. only ~10%)!

<br>

Setting up MNIST data loaders:

In [ ]:
%%save
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import random_split, DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x / 255.)
])

g = torch.Generator().manual_seed(RANDOM_SEED)
ds = MNIST(root=DATASET_DIR, download=False, transform=transform)
ds_train, ds_valid = random_split(ds, [55000, 5000], generator=g)
dl_train = DataLoader(ds_train, batch_size=32, shuffle=True) # (!)
dl_valid = DataLoader(ds_valid, batch_size=32, shuffle=False)

**Remark.** `shuffle=True` is important for SGD training. The model will have low validation score when looping through the samples in the same order during training. This may be due to cyclic behavior in the updates (e.g. cancelling out).